# Spellchecker quest

Хтось наробив помилок у віршах Тараса Шевченка. Ваша задача - виправити ці помилки і прочитати приховане повідомлення.

## Задача

Ви отримаєте тренувальні та тестувальні дані.

Тренувальні дані знаходяться в полі `lab.train_text`. Це звичайний нерозмічений текст. На ньому необхідно натренувати мовну модель. Підійде будь-яка: n-грамна, feed-forward нейронну модель або їх ансамбль. Токенізацію раджу робити по літерах або BPE.

Тестувальні дані знаходиться в полі `lab.test_items`. Приклад одного елемента:

```json
{
  "text": "Співали б прозу, та по ножах,",
  "error_start": 23,
  "error_end": 28,
  "error": "ножах",
  "corrections": [
    "ногах",
    "йотах",
    "єнотах",
    "ножах",
    "нотах"
  ]
}
```

`error_start` та `error_end` вказують на місцезнаходження помилки в тексті (в символах). У цьому прикладі помилкою є `text[23:28]`, тобто слово "ножах".

`corrections` - це список можливих виправлень.

Ваша задача - натренувати мовну модель та за її допомогою обрати правильне виправлення серед запропонованих.


## Приховане повідомлення

Один приклад в `lab.test_items` дає можливість прочитати одну літеру прихованого повідомлення. Для цього знайдіть різницю між літерами слова з помилкою (`error`) та обраним моделлю виправленням. Надрукуйте цю літеру. Якщо слово з помилкою насправді правильне, а таке теж буває, надрукуйте пробіл. Приклади:

```
Error               Correction     Надрукувати
-------------------------------------------
привіт               приліт        л
пні                  поні          о
баллет               балет         л
привіт               привіт        [пробіл]
```

Приховане повідомлення, яке ви побачите в результаті -- це рядок з віршу одного з українських авторів.

Відповідь на квест -- ім'я автора/ки у форматі "Ім'я Прізвище".

Полетіли! 🚀

In [28]:
import lpnlp

lab = lpnlp.start(
    email="sviatoslav.humeniuk.shi.2023@lpnu.ua",
    lab="quest_spellchecker"
)

Удачі!


## Мовна модель

Натренуйте свою мовну модель тут

In [29]:
print(lab.train_text[:52])

ПРИЧИННА

Реве та стогне Дніпр широкий,
Сердитий віт


In [30]:
print(len(lab.train_text))

370249


In [31]:
PAD: str = "\n"

CountsByContext = dict[str, dict[str, int]]
TotalsByContext = dict[str, int]

train_alphabet: set[str] = set(lab.train_text)
vocabulary_size: int = len(train_alphabet) + 1

def train_ngram(
    text: str, context_length: int
) -> tuple[CountsByContext, TotalsByContext]:
    letter_counts: CountsByContext = {}
    context_totals: TotalsByContext = {}

    padded_text = PAD * context_length + text

    for position in range(context_length, len(padded_text)):
        # context = padded_text[position - context_length : position]
        next_letter = padded_text[position]

        for length in range(context_length + 1):
            context = padded_text[position - length : position]

            if context not in letter_counts:
                letter_counts[context] = {}
                context_totals[context] = 0

            if next_letter not in letter_counts[context]:
                letter_counts[context][next_letter] = 0

            letter_counts[context][next_letter] += 1
            context_totals[context] += 1

    return letter_counts, context_totals

print(vocabulary_size)

119


In [32]:
def laplace_probability(
    context: str,
    letter: str,
    letter_counts: CountsByContext,
    context_totals: TotalsByContext
) -> float:
    if context not in letter_counts:
        return 1.0 / vocabulary_size

    times_seen = 0
    if letter in letter_counts[context]:
        times_seen = letter_counts[context][letter]

    return (times_seen + 1) / (context_totals[context] + vocabulary_size)

def witten_bell_probability(
    context: str,
    letter: str,
    letter_counts: CountsByContext,
    context_totals: TotalsByContext
) -> float:
    probability = 1.0 / vocabulary_size

    for length in range(len(context) + 1):
        shorter_context = context[len(context) - length :]

        if shorter_context not in letter_counts:
            break

        times_seen = 0
        if letter in letter_counts[shorter_context]:
            times_seen = letter_counts[shorter_context][letter]

        total = context_totals[shorter_context]
        different_letters = len(letter_counts[shorter_context])

        probability = (times_seen + different_letters * probability) / (total + different_letters)

    return probability

def text_nll(
    text: str,
    context_length: int,
    letter_counts: CountsByContext,
    context_totals: TotalsByContext
) -> float:
    padded_text = PAD * context_length + text
    total_nll = 0.0

    for position in range(context_length, len(padded_text)):
        context = padded_text[position - context_length : position]
        letter = padded_text[position]
        probability = witten_bell_probability(context, letter, letter_counts, context_totals)
        total_nll -= math.log(probability)

    return total_nll

# Читаємо між рядків

In [33]:
import collections

def get_letter(error: str, correction: str) -> str:
    error = error.lower()
    correction = correction.lower()

    if error == correction:
        return " "

    added = collections.Counter(correction) - collections.Counter(error)
    removed = collections.Counter(error) - collections.Counter(correction)
    difference = added or removed

    if sum(difference.values()) != 1:
        raise ValueError(
            f"Очікувалася різниця в одну літеру: {error!r} → {correction!r}"
        )

    return next(iter(difference))

def differs_by_one_letter(error: str, correction: str) -> bool:
    try:
        get_letter(error, correction)
        return True
    except ValueError:
        return False

def get_letter_or_space(error: str, correction: str) -> str:
    try:
        return get_letter(error, correction)
    except ValueError:
        return " "

In [34]:
assert get_letter("привіт", "приліт") == "л"
assert get_letter("пні", "поні") == "о"
assert get_letter("баллет", "балет") == "л"
assert get_letter("Привіт", "привіт") == " "

In [35]:
def choose_correction(
    item: dict,
    context_length: int,
    letter_counts: CountsByContext,
    context_totals: TotalsByContext
) -> str:
    error_word = item["error"]

    candidates: list[str] = []
    for word in item["corrections"]:
        if differs_by_one_letter(error_word, word):
            candidates.append(word)

    if len(candidates) == 0:
        candidates = item["corrections"]

    best_word = candidates[0]
    best_nll = None

    for word in candidates:
        fixed_text = item["text"][: item["error_start"]] + word + item["text"][item["error_end"] :]
        nll = text_nll(fixed_text, context_length, letter_counts, context_totals)

        if best_nll is None or nll < best_nll:
            best_nll = nll
            best_word = word

    return best_word

def solve(
    test_items: list[dict],
    context_length: int,
    letter_counts: CountsByContext,
    context_totals: TotalsByContext
) -> tuple[list[str], str]:
    chosen_words: list[str] = []
    secret_letters: list[str] = []

    for item in test_items:
        word = choose_correction(item, context_length, letter_counts, context_totals)
        chosen_words.append(word)
        secret_letters.append(get_letter_or_space(item["error"], word))

    return chosen_words, "".join(secret_letters)

In [36]:
for ngram_size in (4, 5, 6, 7, 8, 9, 10):
    context_length = ngram_size - 1
    counts, totals = train_ngram(lab.train_text, context_length)
    chosen_words, _ = solve(lab.test_items, context_length, counts, totals)

    print("N =", ngram_size)
    lab.evaluate_accuracy(chosen_words)

N = 4
Відповідь правильна ✅
accuracy = 0.76. Непогано! Спробуй розгадай приховане повідомлення
N = 5
Відповідь правильна ✅
accuracy = 0.80. Непогано! Спробуй розгадай приховане повідомлення
N = 6
Відповідь правильна ✅
accuracy = 0.81. Непогано! Спробуй розгадай приховане повідомлення
N = 7
Відповідь правильна ✅
accuracy = 0.81. Непогано! Спробуй розгадай приховане повідомлення
N = 8
Відповідь правильна ✅
accuracy = 0.82. Непогано! Спробуй розгадай приховане повідомлення
N = 9
Відповідь правильна ✅
accuracy = 0.82. Непогано! Спробуй розгадай приховане повідомлення
N = 10
Відповідь правильна ✅
accuracy = 0.81. Непогано! Спробуй розгадай приховане повідомлення


In [37]:
BEST_NGRAM_SIZE = 5
BEST_CONTEXT_LENGTH = BEST_NGRAM_SIZE - 1

letters_counts, context_totals = train_ngram(lab.train_text, BEST_CONTEXT_LENGTH)
chosen_words, secret_message = solve(lab.test_items, BEST_CONTEXT_LENGTH, letters_counts, context_totals)

lab.evaluate_accuracy(chosen_words)
print(secret_message)

Відповідь правильна ✅
accuracy = 0.80. Непогано! Спробуй розгадай приховане повідомлення
у чнийв еволюцмснев  дух що т ло рве до бою р езза поступпщистя й в лю йін жи е він ще не вмер ніппопів ькії торт риуніктюр мні царсь і мярима і війсьйа муртровані ні парматитллштов н  ні шпі нське ремесло п гвіб йоло щезне звело срнсн пвтер в нищиоживе хоч у дктисячвліт родиввя та  ж вчораюр гповивлязі о в а нійвсиліцйдешійпр ствєтьсяпміцніє і спйшитьстуди дешдн єксловом сильним мов т ибо  міліо и  ве зг обою   ліонимра о йдуть бо се голос дух  чуть


In [38]:
lab.answer("Іван Франко")

Відповідь правильна ✅
Правильно! 🚀 Заповни тепер цю форму, будь ласка: https://tally.so/r/wkl0zZ


# Perplexity

Окрім точності виправлення помилок, оцінимо, наскільки добре ваша модель передбачає текст, якого вона не бачила під час навчання.
Для цього в пакеті є файл `valid.txt`. Його вміст доступний через `lab.valid_text`. Цей текст не входить до тренувальних даних.

Не тренуйте модель або токенізатор на `lab.valid_text`! Використовуйте його лише для оцінювання.

### Що порахувати

Токенізуйте `lab.valid_text` тим самим токенізатором, що й тренувальні дані. Для кожного токена знайдіть імовірність, яку модель надає йому з урахуванням попереднього контексту.

Порахуйте:

1. `N_tokens` — кількість текстових токенів, для яких обчислено ймовірність.
2. `total_NLL` — сумарний від’ємний логарифм їхніх імовірностей:

   ```text
   total_NLL = -Σ ln p(token_i | попередній контекст)
   ```

3. Perplexity на токен:

   ```python
   perplexity = math.exp(total_NLL / N_tokens)
   ```

Менша perplexity означає, що модель краще передбачає цей текст.

Perplexity на токен залежить від токенізації. Наприклад, не можна напряму порівнювати perplexity моделі на літерах із perplexity моделі на BPE-токенах.

Тому додатково порахуйте

4. Perplexity на символ:

```python
N_chars = len(lab.valid_text)
character_perplexity = math.exp(total_NLL / N_chars)
```

У `N_chars` враховуйте весь початковий текст, включно з пробілами та символами нового рядка. Для обох метрик використовуйте той самий `total_NLL`.

Для коректного порівняння моделі мають оцінювати той самий текст. Якщо ви замінюєте невідомі слова на `<unk>`, переводите текст у нижній регістр або викидаєте розділові знаки, ви оцінюєте змінений текст — обов’язково зазначте це у звіті.


In [39]:
import math

valid_nll: float = text_nll(lab.valid_text, BEST_CONTEXT_LENGTH, letters_counts, context_totals)
valid_chars: int = len(lab.valid_text)
valid_tokens: int = valid_chars

valid_token_perplexity: float = math.exp(valid_nll / valid_tokens)
valid_character_perplexity: float = math.exp(valid_nll / valid_chars)

print(valid_nll, valid_chars, valid_tokens, valid_character_perplexity, valid_token_perplexity)

18159.771531293198 10172 10172 5.961192235112069 5.961192235112069


In [40]:
model_and_context = "5-грамна модель з інтерполяцією Віттена-Белла"
tokenization_and_vocab = """
    Символьна токенізація. 118 символів з тренувального набору та 1 для невідомого символу.
    Сумарно 119.
"""
total_nll = 18159.771531293198
n_chars = 10172
n_tokens = 10172
character_perplexity = 5.961192235112069
token_perplexity = 5.961192235112069
processing_notes = """
    Регістр не змінювався, пунктація та пробіли не зазнали змін.
    Словник побудований на основі тренувальних даних, а символ, якого там не було б,
    отримав би частку рівномірної ймовірності 1/119.
    Службовий символ PAD додається N-1 разів на початок тексту лише для контексту перших літер,
    його ймовірність не рахуєтьяс і в n_tokens він не входить.
    BOS реалізовано через PAD, а EOS не використовувались.
    Модель навчена лише на lab.train_text.
    Для порівняння: те саме N=5 зі згладжуванням Лапласа, де коефіцієнт 1,
    дало perplexity 24,05 при тій самій точності виправлень, Віттен–Белл 5,96.
"""

📝  Відправте посилання на цей colab або PDF з ним у відповідне завдання VNS або на email викладачу, що приймає лабораторні
